<a href="https://colab.research.google.com/github/NoaTal1996/Backbone-Graph/blob/main/tf_idf_key_authors.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Imports

In [ ]:
try:
    # Running inside Google Colab → use !pip
    import google.colab
    from google.colab import drive
    in_google_colab = True
    print("Running in Colab → installing packages...")
    !pip install scikit-learn
except ImportError:
    # Not running in Colab
    in_google_colab = False
    pass

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
import pandas as pd

from pathlib import Path

# Loading and Exporting

## Parameters

In [ ]:
dataset_name = "Labeled_Datasets/Ecuador"   # folder path
file_name = "Ecuador_part_5.gzip.parquet"   # file name
dataset_procced_data = "2025_12_01"         # ("%Y_%m_%d")

# Recommended: working directory is: "Backbone-Graph/src"

## Load Data

# TF-IDF Using *sklearn*

In [ ]:

def compute_sklearn_tfidf(df, entity_col, topic_col='topic'):
    """
    Compute TF-IDF over entities grouped by topic.
    Args:
        df (pd.DataFrame): Must contain [topic_col] and [entity_col] (list of entities).
        entity_col (str): Column containing lists of entities per post.
        topic_col (str): Topic label column.
    Returns:
        pd.DataFrame: TF-IDF matrix (rows = topics, cols = entities).
    """

    # 1) Explode entity lists → one row per (topic, entity)
    df_exp = (
        df[[topic_col, entity_col]]
        .explode(entity_col)
        .dropna(subset=[entity_col])
        .astype({entity_col: str})
    )

    # 2) Build "documents" for each topic = all entities of that topic joined into a single string
    topic_docs_df = (
        df_exp.groupby(topic_col)[entity_col]
              .apply(lambda x: " ".join(x))
              .reset_index()
              .rename(columns={entity_col: "doc"})
    )

    # 3) Run TF-IDF
    vec = TfidfVectorizer(token_pattern=r'\S+')  # treat each entity as token
    X = vec.fit_transform(topic_docs_df["doc"])

    tfidf_df = pd.DataFrame(
        X.toarray(),
        columns=vec.get_feature_names_out()
    )

    # 4) Insert topic as first column
    tfidf_df.insert(0, topic_col, topic_docs_df[topic_col].values)

    return tfidf_df


In [ ]:
def get_top_entities(tfidf_df, topic_lable, n=10, topic_col='topic'):
    """
    Return the top n entities with the highest TF-IDF scores for a given topic.
    Args:
        tfidf_df (pd.DataFrame): df ['rows=topics, collums = enteties, cells_values = tf-idf score]
                                 Must include a 'topic' column.
        topic_lable: The topic name or id.
        n (int): Number of top entities to return (default=10).
        topic_col (str): Name of the topic column (default='topic').
    Returns:
        top_entities (pd.Series): Index = entity names, Values = TF-IDF scores (sorted descending).
    """
    # find the row corresponding to the topic
    topic_enteties_df = tfidf_df.loc[tfidf_df[topic_col] == topic_lable]

    # remove the topic column and get entity scores as Series (index = entetity, values = tf-idf score]
    entity_scores_series = topic_enteties_df.drop(columns=[topic_col]).iloc[0]

    # sort entities by score and return top-n
    top_entities = entity_scores_series.sort_values(ascending=False).head(n)

    return top_entities